# ⚡ Transformer Architecture: District-by-District Deep Dive

> **"Generative AI exists because of the transformer."**  
> — Inspired by the Financial Times visual feature ([ig.ft.com/generative-ai](https://ig.ft.com/generative-ai/))

Welcome to the interactive reference notebook for **Transformer**. This guide details each of the 13 districts in the visual simulation, providing both the underlying mathematical formulations and self-contained Python/NumPy implementations.

---


In [ ]:
import numpy as np
import math

# Set random seed for reproducible demonstrations
np.random.seed(42)
print("NumPy version:", np.__version__)


## 🚢 District 1: Tokenizer Docks

### Concept & Mathematics
Computers cannot process raw text strings directly; they require numerical tokens. The **Tokenizer Docks** perform subword tokenization (e.g., Byte-Pair Encoding or WordPiece), mapping text strings $S$ into a sequence of integer token IDs:

$$T = \text{Tokenizer}(S) = [t_1, t_2, \dots, t_N], \quad t_i \in \{0, 1, \dots, V-1\}$$

where $V$ is the vocabulary size.


In [ ]:
# Toy Vocabulary mapping
vocab = {"<PAD>": 0, "the": 1, "transformer": 2, "powers": 3, "generative": 4, "ai": 5, "city": 6, "of": 7, "tokens": 8}
id2vocab = {v: k for k, v in vocab.items()}

def tokenize(text):
    words = text.lower().split()
    return [vocab.get(w, 0) for w in words]

prompt = "the transformer powers generative ai"
token_ids = tokenize(prompt)
print(f"Input text: '{prompt}'")
print(f"Token IDs: {token_ids}")


## 🏭 District 2: Embedding Foundry

### Concept & Mathematics
Integer token IDs lack continuous spatial semantic relationships. The **Embedding Foundry** performs a matrix lookup into an embedding matrix $W_E \in \mathbb{R}^{V \times d_{\text{model}}}$, converting each token ID $t_i$ into a dense vector embedding $e_i \in \mathbb{R}^{d_{\text{model}}}$:

$$e_i = W_E[t_i]$$


In [ ]:
d_model = 12  # Matching the 12-dimensional vector in our visualizer
vocab_size = len(vocab)

# Random initialized embedding matrix
W_E = np.random.randn(vocab_size, d_model)

# Look up embeddings for sequence
embeddings = np.array([W_E[tid] for tid in token_ids])
print("Embeddings shape (seq_len, d_model):", embeddings.shape)
print("Vector for first token ('the'):\n", np.round(embeddings[0], 3))


## 📍 District 3: Positional Beacon

### Concept & Mathematics
Because self-attention processes all tokens simultaneously in parallel, Transformers have no built-in order awareness. The **Positional Beacon** injects sinusoidal positional encodings $PE \in \mathbb{R}^{N \times d_{\text{model}}}$:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i / d_{\text{model}}}}\right)$$

$$X = E + PE$$


In [ ]:
def get_positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i] = math.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                PE[pos, i + 1] = math.cos(pos / (10000 ** (i / d_model)))
    return PE

seq_len = len(token_ids)
PE = get_positional_encoding(seq_len, d_model)
x_pos = embeddings + PE
print("Positional Encoding shape:", PE.shape)
print("Position-encoded vector shape:", x_pos.shape)


## ⚖️ District 4: Pre-Norm Gate

### Concept & Mathematics
To ensure stable training and smooth gradient propagation across deep networks, **Layer Normalization** standardizes vector activations across feature dimensions to zero mean and unit variance:

$$\text{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$$

where $\mu = \frac{1}{d}\sum_{i=1}^d x_i$, $\sigma^2 = \frac{1}{d}\sum_{i=1}^d (x_i - \mu)^2$, and $\gamma, \beta$ are learnable scale/shift vectors.


In [ ]:
def layer_norm(x, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    gamma = np.ones(x.shape[-1])
    beta = np.zeros(x.shape[-1])
    return gamma * (x - mean) / np.sqrt(var + eps) + beta

x_norm = layer_norm(x_pos)
print("Normed mean (approx 0):", np.round(np.mean(x_norm[0]), 5))
print("Normed std (approx 1):", np.round(np.std(x_norm[0]), 5))


## 🎯 District 5: Attention Plaza

### Concept & Mathematics
The core engine of the Transformer! **Multi-Head Self-Attention** projects inputs into Query ($Q$), Key ($K$), and Value ($V$) matrices:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

It then computes scaled dot-product attention with causal masking $M$ (preventing tokens from seeing future tokens):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$


In [ ]:
d_k = d_model
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

Q = np.dot(x_norm, W_Q)
K = np.dot(x_norm, W_K)
V = np.dot(x_norm, W_V)

# Compute raw scores
scores = np.dot(Q, K.T) / math.sqrt(d_k)

# Apply Causal Mask (lower triangular)
causal_mask = np.tril(np.ones((seq_len, seq_len)))
scores = np.where(causal_mask == 1, scores, -1e9)

# Softmax probabilities
exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

attn_output = np.dot(attn_weights, V)
print("Attention Weights matrix (lower-triangular softmax):\n", np.round(attn_weights, 2))
print("Attention output shape:", attn_output.shape)


## 📦 District 6: KV Cache Warehouse

### Concept & Mathematics
During step-by-step token generation (decoding), recalculating $K$ and $V$ for all preceding tokens is computationally wasteful ($O(N^2)$ complexity). The **KV Cache Warehouse** stores computed $K$ and $V$ vectors for past tokens in memory, so only the single new token vector needs to be computed at each step.


In [ ]:
class KVCacheWarehouse:
    def __init__(self):
        self.k_cache = []
        self.v_cache = []
    
    def update(self, new_k, new_v):
        self.k_cache.append(new_k)
        self.v_cache.append(new_v)
        return np.array(self.k_cache), np.array(self.v_cache)

warehouse = KVCacheWarehouse()
for i in range(seq_len):
    k_all, v_all = warehouse.update(K[i], V[i])

print(f"Cached {len(warehouse.k_cache)} token Key/Value pairs in warehouse.")
print("Cached Key shape:", k_all.shape)


## 🌉 District 7: Residual Bridge

### Concept & Mathematics
Deep neural networks suffer from degradation when signals must pass through dozens of layers. The **Residual Bridge** creates skip connections ($X_{\text{out}} = X_{\text{in}} + \text{SubLayer}(X_{\text{in}})$), allowing information to flow directly around sub-layers without distortion.


In [ ]:
x_res1 = x_pos + attn_output
print("Residual Bridge 1 output shape:", x_res1.shape)


## ⚙️ District 8: Feed-Forward Mill

### Concept & Mathematics
While attention mixes information *across tokens*, the **Feed-Forward Mill** processes each token vector independently through a 2-layer MLP with a GELU non-linear activation:

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

$$\text{GELU}(x) \approx 0.5 x \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \left(x + 0.044715 x^3\right)\right)\right)$$


In [ ]:
def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * (x ** 3))))

d_ff = d_model * 2
W1 = np.random.randn(d_model, d_ff) * 0.1
W2 = np.random.randn(d_ff, d_model) * 0.1

x_norm2 = layer_norm(x_res1)
ffn_output = np.dot(gelu(np.dot(x_norm2, W1)), W2)
x_res2 = x_res1 + ffn_output
print("Feed-Forward Mill output shape:", x_res2.shape)


## 🏛️ District 9: Layer Counter Arch

### Concept & Mathematics
Modern language models stack 12 to 96+ identical Transformer blocks in series. The **Layer Counter Arch** tracks hidden state iterations as the vector loops around the repeated layer ring.


In [ ]:
num_layers = 6
h = x_pos.copy()

for layer_idx in range(num_layers):
    # Standard transformer block pass
    h_norm = layer_norm(h)
    q_l, k_l, v_l = np.dot(h_norm, W_Q), np.dot(h_norm, W_K), np.dot(h_norm, W_V)
    sc_l = np.dot(q_l, k_l.T) / math.sqrt(d_k)
    sc_l = np.where(causal_mask == 1, sc_l, -1e9)
    prob_l = np.exp(sc_l - np.max(sc_l, axis=-1, keepdims=True))
    prob_l /= np.sum(prob_l, axis=-1, keepdims=True)
    attn_l = np.dot(prob_l, v_l)
    h = h + attn_l
    
    # FFN
    h_norm2 = layer_norm(h)
    ffn_l = np.dot(gelu(np.dot(h_norm2, W1)), W2)
    h = h + ffn_l
    print(f"Architectural Layer {layer_idx + 1}/{num_layers} complete.")


## 🏟️ District 10: Vocabulary Stadium

### Concept & Mathematics
At the end of the final layer, the hidden state vector of the last token $h_{\text{last}} \in \mathbb{R}^{d_{\text{model}}}$ is projected back to the vocabulary dimension $V$ via the unembedding matrix $W_U \in \mathbb{R}^{d_{\text{model}} \times V}$, producing raw **logits** $Z \in \mathbb{R}^V$:

$$Z = h_{\text{last}} W_U$$
$$P(w_i) = \text{Softmax}(Z)_i = \frac{e^{Z_i}}{\sum_{j=1}^V e^{Z_j}}$$


In [ ]:
W_U = np.random.randn(d_model, vocab_size) * 0.5
last_hidden = h[-1]
logits = np.dot(last_hidden, W_U)

probs = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))
print("Raw Logits for vocabulary:", np.round(logits, 2))
print("Softmax Probabilities:", np.round(probs, 3))


## 🎲 District 11: The Sampler

### Concept & Mathematics
The **Sampler** turns probability distribution $P(w)$ into a single chosen token ID.
1. **Temperature ($T$):** Scales logits ($Z \to Z / T$). $T < 1.0$ makes distributions sharper (focused); $T > 1.0$ flattens distributions (creative).
2. **Top-$P$ (Nucleus) Sampling:** Keeps only the top tokens whose cumulative probability sum reaches threshold $p$.


In [ ]:
def sample(logits, temperature=0.8, top_p=0.9):
    # 1. Temperature scaling
    scaled_logits = logits / temperature
    probs = np.exp(scaled_logits - np.max(scaled_logits))
    probs /= np.sum(probs)
    
    # 2. Sort probabilities descending
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    # 3. Nucleus (Top-p) cutoff
    cumulative_probs = np.cumsum(sorted_probs)
    cutoff_mask = cumulative_probs > top_p
    # Shift right to keep the first index exceeding top_p
    cutoff_mask[1:] = cutoff_mask[:-1]
    cutoff_mask[0] = False
    
    sorted_probs[cutoff_mask] = 0.0
    sorted_probs /= np.sum(sorted_probs)
    
    # 4. Sample token index
    sampled_sorted_idx = np.random.choice(len(sorted_probs), p=sorted_probs)
    sampled_token_id = sorted_indices[sampled_sorted_idx]
    return sampled_token_id

next_token_id = sample(logits, temperature=0.8, top_p=0.9)
print(f"Sampled next token ID: {next_token_id} -> '{id2vocab[next_token_id]}'")


## 📺 District 12 & 🛣️ District 13: Output Plaza & Feedback Highway

### Concept & Mathematics
- **Output Plaza (District 12):** Emits the selected subword token into human-readable text and displays it on the central output Jumbotron.
- **Feedback Highway (District 13):** Appends the new token ID $t_{N+1}$ to the input prompt sequence ($T \gets [T, t_{N+1}]$), triggering the next step of autoregressive generation.


In [ ]:
# Autoregressive loop simulation
current_tokens = token_ids.copy()
print("Starting Prompt:", [id2vocab[i] for i in current_tokens])

for step in range(3):
    # 1. Embed & PE
    E_step = np.array([W_E[tid] for tid in current_tokens])
    PE_step = get_positional_encoding(len(current_tokens), d_model)
    X_step = E_step + PE_step
    
    # 2. Layer pass (simplified)
    h_step = layer_norm(X_step)
    
    # 3. Logits & Sample
    step_logits = np.dot(h_step[-1], W_U)
    new_id = sample(step_logits, temperature=0.8, top_p=0.9)
    current_tokens.append(new_id)
    print(f"Step {step+1}: Emitted '{id2vocab[new_id]}' -> Full text: '{' '.join([id2vocab[i] for i in current_tokens])}'")
